In [0]:
%run "./02_data_preparation"

In [0]:
# ============================================================
# 03.3 VALIDAÇÃO DO DATAFRAME
# ============================================================

# O QUE FAZ:
# Valida se o DataFrame preparado e as informações estruturais necessárias estão disponíveis para execução do Schema Profile.

# COMO FAZ:
# Verifica a existência do DataFrame preparado, das informações do schema e do total de registros calculado anteriormente.

# POR QUE É IMPORTANTE:
# Garante que o Profile seja executado somente após a conclusão correta do Data Preparation, evitando novas ações desnecessárias sobre o DataFrame.

# PERGUNTA RESPONDIDA:
# "O DataFrame preparado possui as informações necessárias para executar o Schema Profile?"

if "df_prepared" not in locals():
    raise ValueError(
        "O DataFrame 'df_prepared' não foi disponibilizado pelo Data Preparation."
    )

if "schema_df" not in locals():
    raise ValueError(
        "O DataFrame 'schema_df' não foi disponibilizado pelo Data Preparation."
    )

if "total_registros" not in locals():
    raise ValueError(
        "A variável 'total_registros' não foi disponibilizada pelo Data Preparation."
    )

if total_registros == 0:
    raise ValueError(
        "O DataFrame preparado não possui registros."
    )

print("DataFrame validado com sucesso.")
print(f"Total de registros: {total_registros}")
print(f"Total de colunas: {len(df_prepared.columns)}")


In [0]:
# ============================================================
# 03.4 INFORMAÇÕES GERAIS
# ============================================================

# O QUE FAZ:
# Organiza as principais informações estruturais do DataFrame que serão utilizadas como referência pelo Schema Profile.

# COMO FAZ:
# Reutiliza as informações já identificadas pelo Data Preparation, evitando nova leitura ou nova contagem do DataFrame.

# POR QUE É IMPORTANTE:
# Centraliza as informações estruturais utilizadas pelo Profile e evita processamento redundante.

# PERGUNTA RESPONDIDA:
# "Qual é a estrutura geral do DataFrame analisado?"

total_columns = len(df_prepared.columns)

general_information = [
    ("Total de linhas", total_registros),
    ("Total de colunas", total_columns),
    ("Numeric columns", len(numeric_columns)),
    ("Text columns", len(text_columns)),
    ("Date columns", len(date_columns)),
    ("Datetime columns", len(datetime_columns)),
    ("Boolean columns", len(boolean_columns)),
    ("Other columns", len(others_columns)),
]

general_info_df = spark.createDataFrame(
    general_information, ["key", "value"]
)

display(general_info_df)

In [0]:
# ============================================================
# 03.5 CONSTRUÇÃO DAS MÉTRICAS
# ============================================================

# O QUE FAZ:
# Constrói as expressões Spark utilizadas para calcular as métricas estruturais de cada coluna.

# COMO FAZ:
# Cria dinamicamente expressões de agregação para preenchimento, NULL, valores vazios e valores distintos.
# As expressões são construídas antes da execução para permitir que o Spark realize as métricas de forma consolidada.

# POR QUE É IMPORTANTE:
# Evita executar uma ação Spark separada para cada coluna, reduzindo a quantidade de leituras e operações no cluster.

# PERGUNTA RESPONDIDA:
# "Quais métricas estruturais serão calculadas para cada coluna?"

metrics_schema = []

for field in df_prepared.schema.fields:

    column = field.name
    ref_column = F.col(column)

    # --------------------------------------------------------
    # VALORES NÃO NULOS
    # --------------------------------------------------------

    metrics_schema.append(
        F.count(ref_column).alias(f"{column}__filled")
    )

    # --------------------------------------------------------
    # VALORES NULL
    # --------------------------------------------------------

    metrics_schema.append(
        F.sum(
            F.when(ref_column.isNull(), 1).otherwise(0)
        ).alias(f"{column}__null")
    )

    # --------------------------------------------------------
    # VALORES DISTINTOS
    # --------------------------------------------------------

    metrics_schema.append(
        F.countDistinct(ref_column).alias(f"{column}__distinct")
    )

    # --------------------------------------------------------
    # VALORES VAZIOS
    # --------------------------------------------------------    

    if isinstance(field.dataType, StringType):
        metrics_schema.append(
            F.sum(
                F.when(
                    ref_column.isNotNull() & 
                    (F.trim(ref_column) == ""), 
                    1
                ).otherwise(0)
            ).alias(f"{column}__empty")
        )

    else:
        metrics_schema.append(
            F.lit(0).cast("long").alias(f"{column}__empty")
        )
    
    print(f"Total built expressions: {len(metrics_schema)}")

In [0]:
# ============================================================
# 03.6 EXECUÇÃO DO PROFILE
# ============================================================

# O QUE FAZ:
# Executa as métricas construídas anteriormente sobre o DataFrame preparado.

# COMO FAZ:
# Utiliza uma única operação de agregação para calcular, simultaneamente, as métricas estruturais de todas as colunas.

# POR QUE É IMPORTANTE:
# Reduz a quantidade de ações Spark e evita realizar uma leitura independente do DataFrame para cada coluna analisada.

# PERGUNTA RESPONDIDA:
# "Quais são os indicadores estruturais observados no DataFrame?"

schema_metrics_row = df_prepared.agg(
    *metrics_schema
).first()

print("Schema profile executed")

In [0]:
# ============================================================
# 03.7 CONSTRUÇÃO DO RESULTADO
# ============================================================

# O QUE FAZ:
# Organiza as métricas agregadas em uma estrutura tabular, apresentando uma linha para cada coluna analisada.

# COMO FAZ:
# Recupera os valores das métricas calculadas na agregação anterior e transforma os resultados em registros estruturados.

# POR QUE É IMPORTANTE:
# Padroniza o resultado do Schema Profile e cria uma estrutura que poderá ser utilizada pelos demais componentes do framework, inclusive pelo Data Quality Score.

# PERGUNTA RESPONDIDA:
# "Quais são as características estruturais de cada coluna?"

results_schema = []

for  field in df_prepared.schema.fields:
    column = field.name

    filled = schema_metrics_row[f"{column}__filled"]
    nulls = schema_metrics_row[f"{column}__null"]
    distinct = schema_metrics_row[f"{column}__distinct"]
    empty = schema_metrics_row[f"{column}__empty"]

    filled_percent = (
        filled / total_registers * 100
        if total_registers > 0
        else 0
    )

    null_percent = (
        nulls / total_registers * 100
        if total_registers > 0
        else 0
    )

    empty_percent = (
        empty / total_registers * 100
        if total_registers > 0
        else 0
    )

    distinct_percent = (
        distinct / total_registers * 100
        if total_registers > 0
        else 0
    )    

    results_schema.append({
        "position": df_prepared.columns.index(column),
        "column": column,
        "detailed_type": field.dataType.simpleString(),
        "null": nulls,
        "filled": filled,
        "empty": empty,
        "distinct": distinct,
        "filled_percent": filled_percent,
        "null_percent": null_percent,
        "empty_percent": empty_percent,
        "distinct_percent": distinct_percent
    })

In [0]:
# ============================================================
# 03.8 DATAFRAME DO SCHEMA PROFILE
# ============================================================

# O QUE FAZ:
# Cria o DataFrame final do Schema Profile a partir das métricas calculadas para cada coluna.

# COMO FAZ:
# Converte os resultados estruturados em um DataFrame Spark e calcula a cardinalidade relativa de cada coluna.

# POR QUE É IMPORTANTE:
# Cria o principal artefato de saída do Schema Profile, disponibilizando uma estrutura padronizada para visualizações, análises posteriores e consolidação do Data Quality Score.

# PERGUNTA RESPONDIDA:
# "Qual é o perfil estrutural completo de cada coluna?"

schema_profile_df = (
    spark.createDataFrame(results_schema)
    .withColumn(
        "cardinality_percent",
        F.when(
            F.col("filled") > 0,
            F.col("distinct") / F.col("filled") * 100
        ).otherwise(F.lit(0.0))
    ).orderBy("position")
)

display(schema_profile_df)

In [0]:
# ============================================================
# 03.9 EXIBIÇÃO
# ============================================================

# O QUE FAZ:
# Exibe o resultado consolidado do Schema Profile.

# COMO FAZ:
# Utiliza a visualização nativa do Databricks sobre o DataFrame Spark produzido na etapa anterior.

# POR QUE É IMPORTANTE:
# Permite validar visualmente as métricas estruturais antes das etapas de visualização e classificação.

# PERGUNTA RESPONDIDA:
# "Como estão distribuídas as principais características estruturais das colunas?"

display(schema_profile_df)

In [0]:
# ============================================================
# 03.10 VISUALIZAÇÃO — PREENCHIMENTO X NULL
# ============================================================

# O QUE FAZ:
# Prepara os indicadores de preenchimento e NULL para visualização comparativa por coluna.

# COMO FAZ:
# Seleciona somente as métricas percentuais necessárias e reorganiza os dados para facilitar a construção de um gráfico de barras no Databricks.

# POR QUE É IMPORTANTE:
# Permite identificar rapidamente colunas com baixo preenchimento ou elevada ocorrência de valores NULL.

# PERGUNTA RESPONDIDA:
# "Quais colunas apresentam problemas de preenchimento?"

filled_null_visualization_df = (
    schema_profile_df
    .select(
        "column",
        "filled_percent",
        "null_percent"
    )
)

display(filled_null_visualization_df)

In [0]:
# ============================================================
# 03.11 VISUALIZAÇÃO — CARDINALIDADE
# ============================================================

# O QUE FAZ:
# Prepara os indicadores de cardinalidade das colunas para visualização.

# COMO FAZ:
# Seleciona a coluna e o percentual de cardinalidade calculado pelo Schema Profile.

# POR QUE É IMPORTANTE:
# A visualização permite identificar colunas com alta ou baixa diversidade relativa de valores.

# PERGUNTA RESPONDIDA:
# "Quais colunas possuem maior ou menor diversidade de valores?"

cardinality_visualization_df = (
    schema_profile_df
    .select(
        "column",
        "distinct",
        "cardinality_percent"
    ).orderBy(F.col("cardinality_percent").desc())
)

display(cardinality_visualization_df)

In [0]:
# ============================================================
# 03.12 CLASSIFICAÇÃO ESTRUTURAL DAS COLUNAS
# ============================================================

# O QUE FAZ:
# Classifica as colunas de acordo com suas características estruturais observadas no Schema Profile.

# COMO FAZ:
# Utiliza indicadores de preenchimento e cardinalidade para identificar características como colunas completas, incompletas, de baixa cardinalidade e de alta cardinalidade.

# POR QUE É IMPORTANTE:
# Transforma métricas estatísticas em informações interpretáveis que poderão ser utilizadas posteriormente pelo Data Quality Score.

# PERGUNTA RESPONDIDA:
# "Qual é a característica estrutural predominante de cada coluna?"

schema_profile_df = (
    schema_profile_df
    .withColumn(
        "filling_classification",
        F.when(F.col("filled_percent") == 100, "COMPLETE")
        .when(F.col("filled_percent") >= 95, "HIGH COMPLETION")
        .when(F.col("filled_percent") >= 80, "MEDIUM COMPLETION")
        .otherwise("LOW COMPLETION")
    )
    .withColumn(
        "cardinality_classification",
        F.when(F.col("cardinality_percent") >= 80, "HIGH CARDINALITY")
        .when(F.col("cardinality_percent") >= 20, "MEDIUM CARDINALITY")
        .otherwise("LOW CARDINALITY")
    )
)

display(schema_profile_df)